# 8.5 实验三：从零训练悬停模型

> **硬件要求**：本实验的简化模型参数量很小（~1万），CPU 即可运行，无需 GPU。Intel Ultra 5 / i5 级别几分钟即可跑完。
>
> **AirSim 配置**：使用 `settings.json`，AirSim 必须在运行。

本节从零开始训练一个简化版世界模型来完成悬停任务。我们将经历完整的训练流程，并亲眼观察一个重要现象——**模型偏差（Model Bias）**，这是模型强化学习中最核心的挑战之一。

![DreamerV3 训练闭环](figures/dreamerv3_training_loop.png)

*图 8-11：本节将完整走一遍这个训练闭环——从数据采集到想象中训练策略*

![悬停任务场景](figures/hover_scene.png)

*图 8-12：悬停任务场景——看似简单，但需要持续精确控制来抵消重力和扰动*

In [ ]:
import sys
sys.path.append('../external-libraries')
sys.path.append('.')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import os, time
from collections import deque

device = torch.device('cpu')  # 简化模型 CPU 足够
print(f"设备: {device}")
print(f"PyTorch 版本: {torch.__version__}")

## 8.5.1 经验回放池

训练世界模型需要大量的交互数据。经验回放池（Replay Buffer）存储历史交互，供训练时随机采样。

In [ ]:
class ReplayBuffer:
    """简单的经验回放池。"""
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch = [self.buffer[i] for i in indices]
        states = torch.tensor(np.array([b[0] for b in batch]), dtype=torch.float32)
        actions = torch.tensor(np.array([b[1] for b in batch]), dtype=torch.float32)
        rewards = torch.tensor(np.array([b[2] for b in batch]), dtype=torch.float32).unsqueeze(1)
        next_states = torch.tensor(np.array([b[3] for b in batch]), dtype=torch.float32)
        dones = torch.tensor(np.array([b[4] for b in batch]), dtype=torch.float32).unsqueeze(1)
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)

buffer = ReplayBuffer()
print(f"回放池容量: {buffer.buffer.maxlen}")

## 8.5.2 数据采集

先用随机策略在 AirSim 中采集一批交互数据，填充回放池。

In [ ]:
import airsim

client = airsim.MultirotorClient()
client.confirmConnection()
DRONE = "Drone1"
TARGET = np.array([0.0, 0.0, -5.0])

def collect_episode(client, policy_fn, buffer, max_steps=200):
    """采集一个 episode 的数据。"""
    client.reset()
    client.enableApiControl(True, vehicle_name=DRONE)
    client.armDisarm(True, vehicle_name=DRONE)
    client.takeoffAsync(vehicle_name=DRONE).join()
    client.moveToPositionAsync(TARGET[0], TARGET[1], TARGET[2], 3, vehicle_name=DRONE).join()
    time.sleep(0.5)

    total_reward = 0
    for step in range(max_steps):
        ms = client.getMultirotorState(vehicle_name=DRONE)
        pos = ms.kinematics_estimated.position
        vel = ms.kinematics_estimated.linear_velocity
        state = np.array([pos.x_val, pos.y_val, pos.z_val,
                          vel.x_val, vel.y_val, vel.z_val], dtype=np.float32)

        action = policy_fn(state)
        client.moveByRollPitchYawrateThrottleAsync(
            float(action[1])*0.3, float(action[0])*0.3,
            float(action[2])*0.5, float(action[3])*0.5+0.5,
            duration=0.1, vehicle_name=DRONE
        ).join()

        ms2 = client.getMultirotorState(vehicle_name=DRONE)
        p2 = ms2.kinematics_estimated.position
        v2 = ms2.kinematics_estimated.linear_velocity
        next_state = np.array([p2.x_val, p2.y_val, p2.z_val,
                               v2.x_val, v2.y_val, v2.z_val], dtype=np.float32)

        dist = np.linalg.norm(next_state[:3] - TARGET)
        collision = client.simGetCollisionInfo(vehicle_name=DRONE)
        done = collision.has_collided or dist > 15
        reward = max(0, 1.0 - dist / 5.0)
        if done:
            reward = -10.0

        buffer.add(state, action, reward, next_state, float(done))
        total_reward += reward
        if done:
            break

    return total_reward, step + 1

# 采集初始数据
random_policy = lambda s: np.random.randn(4).astype(np.float32) * 0.3
print("采集初始数据（随机策略）...")
for ep in range(20):
    r, steps = collect_episode(client, random_policy, buffer)
    if (ep+1) % 5 == 0:
        print(f"  Episode {ep+1}: reward={r:.1f}, steps={steps}, buffer={len(buffer)}")
print(f"数据采集完成，回放池大小: {len(buffer)}")

## 8.5.3 训练世界模型

用采集的数据训练世界模型：学习 (state, action) → (next_state, reward) 的映射。

In [ ]:
from world_model_tools import SimpleWorldModel

world_model = SimpleWorldModel(state_dim=6, action_dim=4, hidden_dim=128).to(device)
wm_optimizer = torch.optim.Adam(world_model.parameters(), lr=3e-4)

wm_losses = []
print("训练世界模型...")
for epoch in range(200):
    states, actions, rewards, next_states, dones = buffer.sample(min(256, len(buffer)))
    states, actions = states.to(device), actions.to(device)
    rewards, next_states = rewards.to(device), next_states.to(device)

    pred_states, pred_rewards = world_model(states, actions)
    loss = nn.MSELoss()(pred_states, next_states) + nn.MSELoss()(pred_rewards, rewards)

    wm_optimizer.zero_grad()
    loss.backward()
    wm_optimizer.step()
    wm_losses.append(loss.item())

    if (epoch+1) % 50 == 0:
        print(f"  Epoch {epoch+1}: loss={loss.item():.6f}")

plt.figure(figsize=(8, 3))
plt.plot(wm_losses)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('世界模型训练损失'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 8.5.4 训练策略网络（Actor）

在世界模型的"想象"中训练策略网络：用世界模型展开虚拟轨迹，优化策略使累计奖励最大化。

In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim=6, action_dim=4, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, action_dim), nn.Tanh(),
        )
    def forward(self, state):
        return self.net(state)

policy = PolicyNetwork().to(device)
policy_optimizer = torch.optim.Adam(policy.parameters(), lr=1e-4)

policy_losses = []
print("在想象中训练策略网络...")
for epoch in range(300):
    states, _, _, _, _ = buffer.sample(min(64, len(buffer)))
    states = states.to(device)

    # 在想象中展开 H 步
    H = 10
    total_reward = torch.zeros(states.shape[0], 1, device=device)
    s = states
    for h in range(H):
        a = policy(s)
        s, r = world_model(s, a)
        total_reward += r * (0.99 ** h)

    loss = -total_reward.mean()  # 最大化奖励 = 最小化负奖励
    policy_optimizer.zero_grad()
    loss.backward()
    policy_optimizer.step()
    policy_losses.append(loss.item())

    if (epoch+1) % 100 == 0:
        print(f"  Epoch {epoch+1}: imagined_reward={-loss.item():.3f}")

plt.figure(figsize=(8, 3))
plt.plot([-l for l in policy_losses])
plt.xlabel('Epoch'); plt.ylabel('Imagined Reward')
plt.title('策略网络训练（想象中的累计奖励）'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 8.5.5 保存 Checkpoint 并推理验证

In [ ]:
# 保存模型
os.makedirs('models/hover_checkpoint', exist_ok=True)
torch.save({
    'world_model': world_model.state_dict(),
    'policy': policy.state_dict(),
}, 'models/hover_checkpoint/model.pt')
print("模型已保存: models/hover_checkpoint/model.pt")

# 用训练好的策略在 AirSim 中推理
def trained_policy(state):
    with torch.no_grad():
        s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        a = policy(s).cpu().numpy()[0]
    return a

print("\n用训练好的策略在 AirSim 中推理验证...")
r, steps = collect_episode(client, trained_policy, ReplayBuffer(1), max_steps=200)
print(f"推理结果: reward={r:.1f}, steps={steps}")

## 8.5.6 观察：模型偏差（Model Bias）

你可能会发现一个反直觉的现象：**训练后的策略在"想象"中表现很好（奖励持续上升），但在真实 AirSim 中反而不如随机策略！**

这就是模型强化学习中最核心的挑战——**模型偏差**：

| 问题 | 原因 | 后果 |
|------|------|------|
| 数据不足 | 只有几百个样本，世界模型学得不准 | 模型预测的"未来"与真实环境不一致 |
| 想象中过拟合 | 策略在不准确的模型中找到了"捷径" | 这些捷径在真实环境中不存在 |
| 分布偏移 | 策略产生的状态超出了训练数据的范围 | 模型在未见过的状态上预测完全错误 |

这就像一个人在梦里学会了飞——梦里的物理规则和现实不同，醒来后发现自己并不会飞。

### 解决方案

DreamerV3 通过以下机制缓解模型偏差：

1. **持续数据采集**：不断用最新策略采集新数据，让世界模型覆盖更多状态
2. **RSSM 架构**：比 MLP 更强大的序列建模能力，预测更准确
3. **短视野想象**：只在想象中展开较短的轨迹（15步），减少误差累积
4. **集成模型**：训练多个世界模型，用它们的分歧来估计不确定性

## 8.5.7 替代方案：MPC（模型预测控制）

与其训练一个策略网络（容易在不准确的模型中过拟合），不如每一步都用世界模型做**在线规划**：采样多个候选动作，用世界模型预测每个动作的短期后果，选择预测奖励最高的那个。

这就是 MPC（Model Predictive Control）的思路——不依赖离线训练的策略，而是实时规划。

In [ ]:
# MPC 策略：每步采样 N 个候选动作，用世界模型评估，选最优
def mpc_policy(state, world_model, n_candidates=100, horizon=3):
    """模型预测控制：在线规划。"""
    state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)  # (1, 6)
    best_action = None
    best_reward = -float('inf')

    # 采样 N 个候选动作
    candidate_actions = torch.randn(n_candidates, 4) * 0.5  # (N, 4)

    with torch.no_grad():
        for i in range(n_candidates):
            action = candidate_actions[i:i+1]  # (1, 4)
            s = state_t.clone()
            total_r = 0
            # 向前看 horizon 步
            for h in range(horizon):
                s, r = world_model(s, action)
                total_r += r.item() * (0.99 ** h)
            if total_r > best_reward:
                best_reward = total_r
                best_action = candidate_actions[i].numpy()

    return np.clip(best_action, -1, 1)

# 用 MPC 策略在 AirSim 中运行
print("MPC 策略（世界模型在线规划）...")
mpc_fn = lambda s: mpc_policy(s, world_model, n_candidates=80, horizon=3)
r_mpc, steps_mpc = collect_episode(client, mpc_fn, ReplayBuffer(1), max_steps=200)

# 对比随机策略
print("随机策略（baseline）...")
r_rand, steps_rand = collect_episode(client, random_policy, ReplayBuffer(1), max_steps=200)

print(f"\n===== 对比结果 =====")
print(f"  随机策略:  reward={r_rand:.1f}, steps={steps_rand}")
print(f"  训练策略:  reward={r:.1f}, steps={steps}")
print(f"  MPC策略:   reward={r_mpc:.1f}, steps={steps_mpc}")
print(f"\nMPC 通过在线规划避免了策略网络的过拟合问题")

## 8.5.8 小结

本节的核心收获：

**训练流程**：数据采集 → 训练世界模型 → 在想象中训练策略 → 推理验证

**关键发现——模型偏差**：
- 策略在"想象"中的奖励持续上升，看起来训练成功了
- 但在真实 AirSim 中，训练后的策略可能不如随机策略
- 原因：世界模型不够准确，策略在不准确的想象中"过拟合"了

**两种使用世界模型的方式**：

| 方式 | 原理 | 对模型精度的要求 |
|------|------|-----------------|
| 策略网络（Actor） | 离线训练，部署时直接输出动作 | 高（需要长视野准确预测） |
| MPC（在线规划） | 每步实时采样+评估候选动作 | 较低（只需短视野预测） |

**从简化模型到 DreamerV3**：本节使用的 MLP 世界模型是最简化的版本。完整的 DreamerV3 使用 RSSM 架构、图像编码器、价值网络、symlog 变换等技术，能够大幅提升世界模型的准确性，从而让策略网络的训练真正有效。